In [0]:
%sql
CREATE OR REPLACE TABLE courier_parcels_delta (
  parcel_id INT,
  customer_name STRING,
  destination_city STRING,
  service_type STRING,
  weight_kg DOUBLE,
  shipping_fee INT,
  delivery_status STRING
)
USING DELTA;

In [0]:
%sql
-- Step 2: Insert sample data
INSERT INTO courier_parcels_delta VALUES
(101, 'Arjun', 'Hyderabad', 'Express', 2.5, 500, 'Delivered'),
(102, 'Meera', 'Mumbai', 'Standard', 4.0, 350, 'In Transit'),
(103, 'Karan', 'Delhi', 'Express', 1.5, 450, 'Pending'),
(104, 'Sneha', 'Pune', 'Standard', 3.0, 300, 'Delivered'),
(105, 'Rohit', 'Chennai', 'Express', 5.0, 800, 'In Transit'),
(106, 'Divya', 'Hyderabad', 'Standard', 2.0, 250, 'Cancelled'),
(107, 'Nikhil', 'Bengaluru', 'Express', 6.0, 950, 'Delivered'),
(108, 'Pooja', 'Mumbai', 'Standard', 1.0, 150, 'Pending'),
(109, 'Vikram', 'Delhi', 'Express', 3.5, 650, 'In Transit'),
(110, 'Anjali', 'Pune', 'Standard', 4.5, 400, 'Cancelled'),
(111, 'Farhan', 'Chennai', 'Express', 2.0, 550, 'Delivered'),
(112, 'Kavya', 'Hyderabad', 'Standard', 5.5, 600, 'Pending'),
(113, 'Manoj', 'Bengaluru', 'Express', 1.5, 400, 'In Transit'),
(114, 'Riya', 'Delhi', 'Standard', 3.0, 300, 'Delivered'),
(115, 'Tarun', 'Mumbai', 'Express', 4.0, 700, 'Pending');

num_affected_rows,num_inserted_rows
15,15


Part A — Creating Delta tables

In [0]:
%sql
-- 1. Display all records from courier_parcels_delta
SELECT * FROM courier_parcels_delta;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
101,Arjun,Hyderabad,Express,2.5,500,Delivered
102,Meera,Mumbai,Standard,4.0,350,In Transit
103,Karan,Delhi,Express,1.5,450,Pending
104,Sneha,Pune,Standard,3.0,300,Delivered
105,Rohit,Chennai,Express,5.0,800,In Transit
106,Divya,Hyderabad,Standard,2.0,250,Cancelled
107,Nikhil,Bengaluru,Express,6.0,950,Delivered
108,Pooja,Mumbai,Standard,1.0,150,Pending
109,Vikram,Delhi,Express,3.5,650,In Transit
110,Anjali,Pune,Standard,4.5,400,Cancelled


In [0]:
%sql
-- 2. Create another Delta table named courier_parcels_copy using CTAS (CREATE TABLE AS SELECT)
CREATE OR REPLACE TABLE courier_parcels_copy 
USING DELTA 
AS SELECT * FROM courier_parcels_delta;

num_affected_rows,num_inserted_rows


In [0]:
# 3. Create a PySpark DataFrame containing three new parcel records and save it as a Delta table named courier_parcels_pyspark
data = [
    (119, 'Suresh', 'Kolkata', 'Express', 3.0, 600, 'Pending'),
    (120, 'Priya', 'Ahmedabad', 'Standard', 2.0, 250, 'Delivered'),
    (121, 'Amit', 'Jaipur', 'Express', 1.5, 400, 'In Transit')
]

columns = ["parcel_id", "customer_name", "destination_city", "service_type", "weight_kg", "shipping_fee", "delivery_status"]

df = spark.createDataFrame(data, columns)
df.write.format("delta").mode("overwrite").saveAsTable("courier_parcels_pyspark")

In [0]:
%sql
-- 4. Verify that all three tables use the Delta format
DESCRIBE DETAIL courier_parcels_delta;
DESCRIBE DETAIL courier_parcels_copy;
DESCRIBE DETAIL courier_parcels_pyspark;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2b772d8a-e582-4622-a726-32c941bd9c49,workspace.default.courier_parcels_pyspark,null,,2026-11-05T09:34:26.996Z,2026-11-05T09:34:29.000Z,List(),List(),1,2426,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Part B — INSERT, UPDATE and DELETE

In [0]:
%sql
-- 5. Insert a new parcel with ID 116, customer Sameer, destination Jaipur, Express service, weight 2.5 kg, shipping fee ₹550, and Pending status
INSERT INTO courier_parcels_delta VALUES
(116, 'Sameer', 'Jaipur', 'Express', 2.5, 550, 'Pending');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- 6. Update the delivery status of parcel 103 to Delivered
UPDATE courier_parcels_delta
SET delivery_status = 'Delivered'
WHERE parcel_id = 103;

num_affected_rows
1


In [0]:
%sql
-- 7. Increase the shipping fee of all Express parcels by ₹100
UPDATE courier_parcels_delta
SET shipping_fee = shipping_fee + 100
WHERE service_type = 'Express';

num_affected_rows
9


In [0]:
%sql
-- 8. Change the destination city of parcel 108 to Hyderabad
UPDATE courier_parcels_delta
SET destination_city = 'Hyderabad'
WHERE parcel_id = 108;

num_affected_rows
1


In [0]:
%sql
-- 9. Delete all parcels with Cancelled status
DELETE FROM courier_parcels_delta
WHERE delivery_status = 'Cancelled';

num_affected_rows
2


In [0]:
%sql
-- 10. Display the remaining records ordered by shipping fee in descending order
SELECT * 
FROM courier_parcels_delta
ORDER BY shipping_fee DESC;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
107,Nikhil,Bengaluru,Express,6.0,1050,Delivered
105,Rohit,Chennai,Express,5.0,900,In Transit
115,Tarun,Mumbai,Express,4.0,800,Pending
109,Vikram,Delhi,Express,3.5,750,In Transit
116,Sameer,Jaipur,Express,2.5,650,Pending
111,Farhan,Chennai,Express,2.0,650,Delivered
101,Arjun,Hyderabad,Express,2.5,600,Delivered
112,Kavya,Hyderabad,Standard,5.5,600,Pending
103,Karan,Delhi,Express,1.5,550,Delivered
113,Manoj,Bengaluru,Express,1.5,500,In Transit


Part C — MERGE and UPSERT (SCD Type 1)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parcels AS
SELECT *
FROM VALUES
(102, 'Meera', 'Mumbai', 'Express', 4.0, 650, 'Delivered'),
(105, 'Rohit', 'Chennai', 'Express', 5.0, 900, 'Delivered'),
(117, 'Ayesha', 'Kolkata', 'Standard', 2.0, 300, 'Pending'),
(118, 'Rahul', 'Jaipur', 'Express', 3.0, 600, 'In Transit')
AS incoming(
  parcel_id,
  customer_name,
  destination_city,
  service_type,
  weight_kg,
  shipping_fee,
  delivery_status
);

In [0]:
%sql
-- 11. Display all records from incoming_parcels
SELECT * FROM incoming_parcels;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered
105,Rohit,Chennai,Express,5.0,900,Delivered
117,Ayesha,Kolkata,Standard,2.0,300,Pending
118,Rahul,Jaipur,Express,3.0,600,In Transit


In [0]:
%sql
-- 12. Identify which incoming parcel IDs already exist in the target Delta table
SELECT incoming.parcel_id, incoming.customer_name
FROM incoming_parcels incoming
INNER JOIN courier_parcels_delta target
ON incoming.parcel_id = target.parcel_id;

parcel_id,customer_name
102,Meera
105,Rohit


In [0]:
%sql
-- 13. Identify which incoming parcel IDs are new
SELECT incoming.parcel_id, incoming.customer_name
FROM incoming_parcels incoming
LEFT JOIN courier_parcels_delta target
ON incoming.parcel_id = target.parcel_id
WHERE target.parcel_id IS NULL;

parcel_id,customer_name
117,Ayesha
118,Rahul


In [0]:
%sql
-- 14. Perform a MERGE operation that updates all columns for existing parcels and inserts new parcels
MERGE INTO courier_parcels_delta AS target
USING incoming_parcels AS source
ON target.parcel_id = source.parcel_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
-- 15. Verify that parcels 102 and 105 were updated
SELECT * 
FROM courier_parcels_delta 
WHERE parcel_id IN (102, 105);

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
102,Meera,Mumbai,Express,4.0,650,Delivered
105,Rohit,Chennai,Express,5.0,900,Delivered


In [0]:
%sql
-- 16. Verify that parcels 117 and 118 were inserted
SELECT * 
FROM courier_parcels_delta 
WHERE parcel_id IN (117, 118);

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
117,Ayesha,Kolkata,Standard,2.0,300,Pending
118,Rahul,Jaipur,Express,3.0,600,In Transit


Part D — Delta table internals and Time Travel

In [0]:
%sql
-- 17. Execute DESCRIBE DETAIL and identify the table's format, storage location, number of files, and size
DESCRIBE DETAIL courier_parcels_delta;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,23bd5edc-6256-47e2-b106-517f21b99baa,workspace.default.courier_parcels_delta,null,,2026-11-05T09:32:43.981Z,2026-11-05T09:38:45.000Z,List(),List(),1,3587,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-88f44cc4-a30e-4635-a7bd-d832973cf185, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-6938d68b-a84a-4858-8a2e-7d126c57e1ed)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- 18. Execute DESCRIBE HISTORY and identify the operations performed on the table
DESCRIBE HISTORY courier_parcels_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-11-05T09:38:45.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2469590266806038),4e411ef1-731c-47fe-b259-bf96f60ba65c,1105-092847-pqsfsfw8-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 15198, p25FileSize -> 3587, numDeletionVectorsRemoved -> 1, minFileSize -> 3587, numAddedFiles -> 1, maxFileSize -> 3587, p75FileSize -> 3587, p50FileSize -> 3587, numAddedBytes -> 3587)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
9,2026-11-05T09:38:43.000Z,70541099100131,jersilgm@gmail.com,MERGE,"Map(predicate -> [""(parcel_id#15418 = parcel_id#15397)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2469590266806038),4e411ef1-731c-47fe-b259-bf96f60ba65c,1105-092847-pqsfsfw8-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 11666, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4261, materializeSourceTimeMs -> 288, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1179, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2706)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-05T09:37:27.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2469590266806038),e0831ce3-50e5-4ad4-ac8b-e4eeaa51e1b0,1105-092847-pqsfsfw8-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6532, p25FileSize -> 3532, numDeletionVectorsRemoved -> 1, minFileSize -> 3532, numAddedFiles -> 1, maxFileSize -> 3532, p75FileSize -> 3532, p50FileSize -> 3532, numAddedBytes -> 3532)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-05T09:37:24.000Z,70541099100131,jersilgm@gmail.com,DELETE,"Map(predicate -> [""(delivery_status#14465 = Cancelled)""])",null,List(2469590266806038),e0831ce3-50e5-4ad4-ac8b-e4eeaa51e1b0,1105-092847-pqsfsfw8-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1202, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 711, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 490)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-05T09:37:13.000Z,70541099100131,jersilgm@gmail.com,UPDATE,"Map(predicate -> [""(parcel_id#13860 = 108)""])",null,List(2469590266806038),165618a9-b0a2-403b-9d69-0419a24f9fc1,1105-092847-pqsfsfw8-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2555, numDeletionVectorsUpdated -> 0, scanTimeMs -> 966, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2984, rewriteTimeMs -> 1588)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-05T09:37:05.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2469590266806038),5f8a5080-5b41-4611-a98b-deabcbf58ae2,1105-092847-pqsfsfw8-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRe

In [0]:
%sql
-- 19. Find the table version immediately before the MERGE operation
SELECT version, timestamp, operation 
FROM (DESCRIBE HISTORY courier_parcels_delta)
ORDER BY version DESC;

version,timestamp,operation
10,2026-11-05T09:38:45.000Z,OPTIMIZE
9,2026-11-05T09:38:43.000Z,MERGE
8,2026-11-05T09:37:27.000Z,OPTIMIZE
7,2026-11-05T09:37:24.000Z,DELETE
6,2026-11-05T09:37:13.000Z,UPDATE
5,2026-11-05T09:37:05.000Z,OPTIMIZE
4,2026-11-05T09:37:03.000Z,UPDATE
3,2026-11-05T09:36:53.000Z,UPDATE
2,2026-11-05T09:36:38.000Z,WRITE
1,2026-11-05T09:33:02.000Z,WRITE


In [0]:
%sql
-- 20. Use VERSION AS OF to retrieve the records before the MERGE
SELECT * 
FROM courier_parcels_delta VERSION AS OF 8;

parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
108,Pooja,Hyderabad,Standard,1.0,150,Pending
102,Meera,Mumbai,Standard,4.0,350,In Transit
104,Sneha,Pune,Standard,3.0,300,Delivered
112,Kavya,Hyderabad,Standard,5.5,600,Pending
114,Riya,Delhi,Standard,3.0,300,Delivered
101,Arjun,Hyderabad,Express,2.5,600,Delivered
105,Rohit,Chennai,Express,5.0,900,In Transit
107,Nikhil,Bengaluru,Express,6.0,1050,Delivered
109,Vikram,Delhi,Express,3.5,750,In Transit
111,Farhan,Chennai,Express,2.0,650,Delivered


In [0]:
%sql
-- 21. Compare parcel 102 in the previous version with its current record

SELECT 'Old Version' AS data_state, * 
FROM courier_parcels_delta VERSION AS OF 8 
WHERE parcel_id = 102
UNION ALL
SELECT 'Current Version' AS data_state, * 
FROM courier_parcels_delta 
WHERE parcel_id = 102;

data_state,parcel_id,customer_name,destination_city,service_type,weight_kg,shipping_fee,delivery_status
Old Version,102,Meera,Mumbai,Standard,4.0,350,In Transit
Current Version,102,Meera,Mumbai,Express,4.0,650,Delivered


Part E — OPTIMIZE

In [0]:
%sql
-- 22. Execute OPTIMIZE on courier_parcels_delta
OPTIMIZE courier_parcels_delta;

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793871700116, 1793871703149, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


In [0]:
%sql
-- 23. Run DESCRIBE DETAIL again and compare the number of files before and after optimization
DESCRIBE DETAIL courier_parcels_delta;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,23bd5edc-6256-47e2-b106-517f21b99baa,workspace.default.courier_parcels_delta,null,,2026-11-05T09:32:43.981Z,2026-11-05T09:38:45.000Z,List(),List(),1,3587,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-88f44cc4-a30e-4635-a7bd-d832973cf185, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-6938d68b-a84a-4858-8a2e-7d126c57e1ed)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- 24. Execute OPTIMIZE courier_parcels_delta ZORDER BY (destination_city) if the table does not use liquid clustering
OPTIMIZE courier_parcels_delta 
ZORDER BY (destination_city);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3587), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793871721000, 1793871723782, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, null, null, 0)"


In [0]:
%sql
-- 25. Display the transaction history again and identify the optimization operations
DESCRIBE HISTORY courier_parcels_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-11-05T09:38:45.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2469590266806038),4e411ef1-731c-47fe-b259-bf96f60ba65c,1105-092847-pqsfsfw8-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 15198, p25FileSize -> 3587, numDeletionVectorsRemoved -> 1, minFileSize -> 3587, numAddedFiles -> 1, maxFileSize -> 3587, p75FileSize -> 3587, p50FileSize -> 3587, numAddedBytes -> 3587)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
9,2026-11-05T09:38:43.000Z,70541099100131,jersilgm@gmail.com,MERGE,"Map(predicate -> [""(parcel_id#15418 = parcel_id#15397)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2469590266806038),4e411ef1-731c-47fe-b259-bf96f60ba65c,1105-092847-pqsfsfw8-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 11666, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4261, materializeSourceTimeMs -> 288, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1179, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2706)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
8,2026-11-05T09:37:27.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2469590266806038),e0831ce3-50e5-4ad4-ac8b-e4eeaa51e1b0,1105-092847-pqsfsfw8-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 6532, p25FileSize -> 3532, numDeletionVectorsRemoved -> 1, minFileSize -> 3532, numAddedFiles -> 1, maxFileSize -> 3532, p75FileSize -> 3532, p50FileSize -> 3532, numAddedBytes -> 3532)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
7,2026-11-05T09:37:24.000Z,70541099100131,jersilgm@gmail.com,DELETE,"Map(predicate -> [""(delivery_status#14465 = Cancelled)""])",null,List(2469590266806038),e0831ce3-50e5-4ad4-ac8b-e4eeaa51e1b0,1105-092847-pqsfsfw8-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1202, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 711, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 490)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
6,2026-11-05T09:37:13.000Z,70541099100131,jersilgm@gmail.com,UPDATE,"Map(predicate -> [""(parcel_id#13860 = 108)""])",null,List(2469590266806038),165618a9-b0a2-403b-9d69-0419a24f9fc1,1105-092847-pqsfsfw8-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2555, numDeletionVectorsUpdated -> 0, scanTimeMs -> 966, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2984, rewriteTimeMs -> 1588)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-05T09:37:05.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2469590266806038),5f8a5080-5b41-4611-a98b-deabcbf58ae2,1105-092847-pqsfsfw8-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRe